In [9]:
%pip install -q pypdf numpy faiss-cpu google-genai

Note: you may need to restart the kernel to use updated packages.


In [10]:
import os
import time
import numpy as np
import faiss
from pypdf import PdfReader
from google import genai
from google.genai import types
import getpass

print("Libraries imported successfully!")

Libraries imported successfully!


In [11]:
# Enter your Gemini API key privately.
# Do NOT hard-code the key in the notebook.

API_KEY = getpass.getpass("Enter your Gemini API key: ")
client = genai.Client(api_key=API_KEY)

EMBEDDING_MODEL = "gemini-embedding-001"
GENERATION_MODEL = "gemini-3.8-flash"

print("Gemini client created successfully!")

Enter your Gemini API key:  ········


Gemini client created successfully!


In [12]:
def load_pdf(file_path):
    pages_data = []
    reader = PdfReader(file_path)

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        pages_data.append({
            "source": file_path,
            "page": page_number,
            "text": text
        })

    return pages_data


def load_multiple_pdfs(file_paths):
    all_pages = []

    for file_path in file_paths:
        all_pages.extend(load_pdf(file_path))

    return all_pages


def chunk_pages(pages_data, chunk_size=150, overlap=30):
    chunks = []

    for page_data in pages_data:
        words = page_data["text"].split()
        start = 0

        while start < len(words):
            end = start + chunk_size
            chunk_text = " ".join(words[start:end])

            if chunk_text.strip():
                chunks.append({
                    "id": len(chunks),
                    "text": chunk_text,
                    "source": page_data["source"],
                    "page": page_data["page"]
                })

            start += chunk_size - overlap

    return chunks


pdf_files = ["DBMS_Notes.pdf"]

pages_data = load_multiple_pdfs(pdf_files)
all_chunks = chunk_pages(pages_data, chunk_size=150, overlap=30)

print("PDF pages loaded:", len(pages_data))
print("Total chunks created:", len(all_chunks))

PDF pages loaded: 1
Total chunks created: 1


In [13]:
# Create document embeddings for RAG

chunk_embeddings = []

for i, chunk in enumerate(all_chunks):
    result = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=chunk["text"],
        config=types.EmbedContentConfig(
            task_type="RETRIEVAL_DOCUMENT"
        )
    )

    chunk_embeddings.append(result.embeddings[0].values)

embedding_matrix = np.array(chunk_embeddings, dtype="float32")

# Normalize because we will use cosine similarity via inner product.
faiss.normalize_L2(embedding_matrix)

print("Embeddings generated successfully!")
print("Embedding matrix shape:", embedding_matrix.shape)

Embeddings generated successfully!
Embedding matrix shape: (1, 3072)


In [14]:
# Build FAISS vector index

dimension = embedding_matrix.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embedding_matrix)

print("FAISS index created successfully!")
print("Total vectors:", index.ntotal)
print("Vector dimension:", dimension)

FAISS index created successfully!
Total vectors: 1
Vector dimension: 3072


In [21]:
chat_history = []


def create_question_embedding(question):
    result = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=question,
        config=types.EmbedContentConfig(
            task_type="RETRIEVAL_QUERY"
        )
    )

    question_embedding = np.array(
        [result.embeddings[0].values],
        dtype="float32"
    )

    faiss.normalize_L2(question_embedding)
    return question_embedding


def generate_answer(prompt):
    try:
        response = client.models.generate_content(
            model="gemini-3.8-flash",
            contents=prompt
        )
        return response.text

    except Exception as e:
        error_message = str(e)

        if "429" in error_message or "RESOURCE_EXHAUSTED" in error_message:
            return (
                "Gemini API quota has been exceeded. "
                "Please try again after the quota resets."
            )

        return f"Gemini API error: {error_message}"


def ask(question, k=2):
    if not all_chunks or index.ntotal == 0:
        raise ValueError("No PDF chunks or FAISS vectors are available.")

    k = min(k, index.ntotal)

    question_embedding = create_question_embedding(question)
    scores, indices = index.search(question_embedding, k)

    retrieved_context = ""
    sources = []

    for i in range(k):
        chunk_index = int(indices[0][i])
        chunk = all_chunks[chunk_index]
        score = float(scores[0][i])

        retrieved_context += (
            f"[Source: {chunk['source']}, Page: {chunk['page']}]\n"
            f"{chunk['text']}\n\n"
        )

        sources.append({
            "source": chunk["source"],
            "page": chunk["page"],
            "score": score
        })

    history_text = ""
    for message in chat_history[-4:]:
        history_text += f"{message['role'].title()}: {message['content']}\n"

    prompt = f"""
You are CampusGPT, an AI learning assistant for students.

Answer the student's question ONLY using the information provided
in the retrieved context from the uploaded PDF documents.

If the answer is not present in the context, reply exactly with:

"I couldn't find that information in the uploaded PDF(s)."

Do not use outside knowledge.

For facts taken from the context, cite the source and page like:
(Source: file.pdf, Page: 3)

Recent conversation:
{history_text}

Retrieved context:
{retrieved_context}

Student Question:
{question}

Answer clearly and in simple language.
"""

    answer = generate_answer(prompt)

    chat_history.append({"role": "user", "content": question})
    chat_history.append({"role": "assistant", "content": answer})

    return answer, sources


print("CampusGPT functions created successfully!")

CampusGPT functions created successfully!


In [22]:
# Test 1
answer, sources = ask("What is a database management system?")

print("CampusGPT Answer:")
print(answer)

print("\nSources:")
for source in sources:
    print(
        f"Source: {source['source']}, "
        f"Page: {source['page']}, "
        f"Similarity Score: {source['score']:.4f}"
    )

CampusGPT Answer:
Gemini API quota has been exceeded. Please try again after the quota resets.

Sources:
Source: DBMS_Notes.pdf, Page: 1, Similarity Score: 0.7666


In [23]:
answer, sources = ask("What is a database management system?")

print("CampusGPT Answer:")
print(answer)

print("\nSources:")
for source in sources:
    print(source)

CampusGPT Answer:
Gemini API quota has been exceeded. Please try again after the quota resets.

Sources:
{'source': 'DBMS_Notes.pdf', 'page': 1, 'score': 0.7666364312171936}


In [24]:
question = "What is a database management system?"

question_result = client.models.embed_content(
    model="gemini-embedding-001",
    contents=question,
    config={
        "task_type": "RETRIEVAL_QUERY"
    }
)

question_embedding = np.array(
    [question_result.embeddings[0].values],
    dtype="float32"
)

faiss.normalize_L2(question_embedding)

scores, indices = index.search(question_embedding, 2)

print("Retrieved Context:\n")

for i in range(2):
    chunk_index = indices[0][i]

    print(f"Source: {all_chunks[chunk_index]['source']}")
    print(f"Page: {all_chunks[chunk_index]['page']}")
    print(f"Similarity: {scores[0][i]:.4f}")
    print(all_chunks[chunk_index]["text"])
    print("-" * 70)

Retrieved Context:

Source: DBMS_Notes.pdf
Page: 1
Similarity: 0.7666
Database Management System A Database Management System (DBMS) is software used to create, manage, store and retrieve data. Examples of DBMS include MySQL, Oracle and PostgreSQL. A database is an organized collection of related data. SQL is commonly used to interact with relational databases.
----------------------------------------------------------------------
Source: DBMS_Notes.pdf
Page: 1
Similarity: -340282346638528859811704183484516925440.0000
Database Management System A Database Management System (DBMS) is software used to create, manage, store and retrieve data. Examples of DBMS include MySQL, Oracle and PostgreSQL. A database is an organized collection of related data. SQL is commonly used to interact with relational databases.
----------------------------------------------------------------------


In [2]:
from google import genai
import getpass

API_KEY = getpass.getpass("Enter your Gemini API key: ")
client = genai.Client(api_key=API_KEY)

Enter your Gemini API key:  ········


In [3]:
# Check available Gemini models

for model in client.models.list():
    if "generateContent" in model.supported_actions:
        print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/lyria-3.5
models/gemini-3.1-flash-tts-preview
models/

In [4]:
test_response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="Say hello in one sentence."
)

print(test_response.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Hello, I hope you are having a wonderful day!


In [1]:
from google import genai
import getpass

API_KEY = getpass.getpass("Enter your Gemini API key: ")

client = genai.Client(api_key=API_KEY)

print("Available Gemini models:\n")

for model in client.models.list():
    print(model.name)

Enter your Gemini API key:  ········


Available Gemini models:

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/lyria-3.5
models/gemini-3.1

In [2]:
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Say hello in one short sentence."
)

print(response.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Hello and welcome!
